In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

URL = "https://en.wikipedia.org/wiki/List_of_Olympic_Games"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(URL, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

# Get all wikitable tables
tables = soup.find_all("table", {"class": "wikitable"})

# First two tables = Summer & Winter
summer_table = tables[0]
winter_table = tables[1]


# -----------------------------
# Helper functions
# -----------------------------
def extract_year(text):
    match = re.search(r"\b(18|19|20|21)\d{2}\b", text)
    return int(match.group()) if match else None


def clean_text(text):
    text = re.sub(r"\[.*?\]", "", text)
    return text.strip()


def clean_city(city):
    city = clean_text(city)
    city = re.sub(r"\s*(–|-|,| and )\s*", ";", city)
    return city


def detect_cancelled(text):
    text = text.lower()
    return any(k in text for k in ["cancelled", "canceled", "not held"])


def parse_table(table, season):
    data = []
    rows = table.find_all("tr")

    for row in rows:
        cells = row.find_all(["td", "th"])

        if len(cells) < 3:
            continue

        texts = [cell.get_text(" ", strip=True) for cell in cells]

        full_text = " ".join(texts)

        # Extract year from entire row
        year = extract_year(full_text)

        # Heuristic: first non-year text = city
        city = None
        country = None

        for t in texts:
            if not city and not re.search(r"\d{4}", t):
                city = clean_city(t)
                continue

            if city and not country:
                country = clean_text(t)
                break

        if not year:
            continue

        data.append({
            "year": year,
            "city": city,
            "country": country,
            "season": season,
            "games_label": f"{city} {year}" if city else f"Olympics {year}",
            "is_cancelled": detect_cancelled(full_text),
            "num_hosts": len(city.split(";")) if city else 0
        })

    return data


# -----------------------------
# Build dataset
# -----------------------------
data = []
data.extend(parse_table(summer_table, "Summer"))
data.extend(parse_table(winter_table, "Winter"))

df = pd.DataFrame(data)

df = df.drop_duplicates().sort_values(["year", "season"]).reset_index(drop=True)

# Save
df.to_csv("olympic_hosts_clean.csv", index=False)

print("✅ Rows:", len(df))
df.head()

✅ Rows: 74


,year,city,country,season,games_label,is_cancelled,num_hosts
0,1896,Athens,Greece,Summer,Athens 1896,False,1
1,1896,Athens,Greece,Winter,Athens 1896,False,1
2,1900,Paris,France,Summer,Paris 1900,False,1
3,1900,Paris,France,Winter,Paris 1900,False,1
4,1904,St. Louis,United States,Summer,St. Louis 1904,False,1


In [13]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

URL = "https://en.wikipedia.org/wiki/List_of_Olympic_Games_host_cities"

headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(URL, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

tables = soup.find_all("table", class_="wikitable")

# 🔥 Find correct table dynamically
target_table = None

for table in tables:
    headers = [th.get_text(strip=True).lower() for th in table.find_all("th")]

    if any("city" in h for h in headers) and any("country" in h for h in headers):
        target_table = table
        break

if target_table is None:
    raise ValueError("❌ Could not find correct table")

print("✅ Table found")

# -----------------------------
# Helpers
# -----------------------------
def clean_text(text):
    text = re.sub(r"\[.*?\]", "", text)
    return text.strip()

def extract_year(text):
    m = re.search(r"\b(18|19|20|21)\d{2}\b", text)
    return int(m.group()) if m else None

def clean_city(city):
    city = clean_text(city)
    city = re.sub(r"\s*(–|-|,| and )\s*", ";", city)
    return city

def count_hosts(city):
    return len(city.split(";")) if city else 0

def detect_status(text):
    text = text.lower()

    if "†" in text or "cancelled" in text:
        return "Cancelled"
    if "§" in text or "postponed" in text:
        return "Postponed"
    return "Held"

def infer_season(year):
    """
    Robust inference:
    - After 1994: Winter and Summer alternate every 2 years
    - Before 1994: both happened same year → fallback to city knowledge
    """

    if year >= 1994:
        # Winter years: 1994, 1998, 2002, ...
        if (year - 1994) % 4 == 0:
            return "Winter"
        else:
            return "Summer"

    else:
        # Pre-1994: both Summer + Winter in same year
        # We infer using known winter host patterns (cold regions heuristic)

        winter_keywords = [
            "norway", "canada", "switzerland", "austria",
            "japan", "italy", "france", "germany"
        ]

        return "Winter" if any(k in str(country).lower() for k in winter_keywords) else "Summer"


# -----------------------------
# Parse table
# -----------------------------
records = []

rows = target_table.find_all("tr")[1:]

for row in rows:
    cells = row.find_all(["td", "th"])
    if len(cells) < 5:
        continue

    texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
    full_text = " ".join(texts)

    year = extract_year(full_text)
    if not year:
        continue

    city = clean_city(texts[1])
    country = texts[2]

    # 🔥 KEY FIX: detect which column has data
    summer_col = texts[3]
    winter_col = texts[4]

    if summer_col and not winter_col:
        season = "Summer"
    elif winter_col and not summer_col:
        season = "Winter"
    else:
        continue  # skip ambiguous rows

    status = detect_status(full_text)

    records.append({
        "year": year,
        "host_city": city,
        "host_country": country,
        "season": season,
        "status": status,
        "is_cancelled": status == "Cancelled",
        "is_postponed": status == "Postponed",
        "num_hosts": count_hosts(city),
        "games_label": f"{city} {year}"
    })

df = pd.DataFrame(records)

if df.empty:
    raise ValueError("❌ Still empty — print headers for debugging")

df = df.drop_duplicates().sort_values(["year", "season"]).reset_index(drop=True)

df.to_csv("olympic_hosts_clean.csv", index=False)

print("✅ Rows:", len(df))
df.head()

✅ Table found
✅ Rows: 22


,year,host_city,host_country,season,status,is_cancelled,is_postponed,num_hosts,games_label
0,1924,Chamonix,France,Summer,Held,False,False,1,Chamonix 1924
1,1924,Paris,S008 VIII,Winter,Held,False,False,1,Paris 1924
2,1928,St. Moritz,Switzerland,Summer,Held,False,False,1,St. Moritz 1928
3,1928,Amsterdam,Netherlands,Summer,Held,False,False,1,Amsterdam 1928
4,1932,Los Angeles,S010 X,Winter,Held,False,False,1,Los Angeles 1932
